In [5]:
#Importa polars como pl
import polars as pl
# Importa numpy como np
import numpy as np
# Importamos modulos de Python para manipular rutas de archivos
import sys
from pathlib import Path

# Agrega el directorio la raíz del proyecto para importar los scripts .py
sys.path.append("..")
# Importa la ruta hacia data/data-processed/ definida en tu script
from config.rutas import RUTA_DATA_PROCESSED

# Cargar dataset parquet procesado con Polars
df = pl.read_parquet(RUTA_DATA_PROCESSED / "endireh_limpio.parquet")
# Imprime el total de filas y columnas: (105,753 7) para verificar que se haya
# importado correctamente
print("Dimensiones dataset:", df.shape)
# Crea una tabla con las primeras 5 filas del dataset para verificar el formato esperado.
df.head()

Dimensiones dataset: (105753, 7)


edad_primer_union,num_hijos,nivel_escolaridad,estado_civil_desc,factor_expansion,nom_entidad,sufrio_violencia_pareja
str,str,str,str,f64,str,i32
null,"""3.0""","""A1""","""Casada""",2096.0,"""VERACRUZ DE IGNACIO DE LA LLAV…",0
"""4.0""","""1.0""","""B1""","""Separada""",2253.0,"""MÃÂXICO""",1
"""20.0""","""3.0""","""B2""","""Viuda""",347.0,"""GUANAJUATO""",0
null,"""3.0""","""A1""","""Casada""",602.0,"""JALISCO""",0
null,null,"""C1""","""Soltera""",1139.0,"""GUANAJUATO""",0


In [22]:
#Importa polars como pl
import polars as pl

# Creamos tabla para comparar entre cuantitativa y cualitativa y su justtificación
# cada elemento representa una fila de la tabla del Paso 5.
clasificacion = [
    {"Variable": "edad_primer_union", "Tipo de Dato": "Cuantitativa", "Justificación": "Años de la mujer al unirse por primera vez."},
    {"Variable": "num_hijos", "Tipo de Dato": "Cuantitativa", "Justificación": "Cuenta el número total de hijos nacidos."},
    {"Variable": "nom_entidad", "Tipo de Dato": "Cualitativa", "Justificación": "Categoría de los 32 estados."},
    {"Variable": "nivel_escolaridad", "Tipo de Dato": "Cualitativa", "Justificación": "Niveles de estudios con jerarquía: Sin escolaridad, primaria, secundaria, etc."},
    {"Variable": "estado_civil_desc", "Tipo de Dato": "Cualitativa", "Justificación": "Estado conyugal de mujer: casada, soltera, divorciada, etc."},
    {"Variable": "sufrio_violencia_pareja", "Tipo de Dato": "Cualitativa", "Justificación": "Indica si sufrio violencia. 1 = Si, 0 = No."},
    {"Variable": "factor_expansion", "Tipo de Dato": "Cuantitativa", "Justificación": "Representa el número de mujeres que equivale la observación."}
]

# Convierte la lista de diccionarios en DataFrame de Polars.
df_clas = pl.DataFrame(clasificacion)
# Imprime la tabla (Dataframe)
print(df_clas)

shape: (7, 3)
┌─────────────────────────┬──────────────┬─────────────────────────────────┐
│ Variable                ┆ Tipo de Dato ┆ Justificación                   │
│ ---                     ┆ ---          ┆ ---                             │
│ str                     ┆ str          ┆ str                             │
╞═════════════════════════╪══════════════╪═════════════════════════════════╡
│ edad_primer_union       ┆ Cuantitativa ┆ Años de la mujer al unirse por… │
│ num_hijos               ┆ Cuantitativa ┆ Cuenta el número total de hijo… │
│ nom_entidad             ┆ Cualitativa  ┆ Categoría de los 32 estados.    │
│ nivel_escolaridad       ┆ Cualitativa  ┆ Niveles de estudios con jerarq… │
│ estado_civil_desc       ┆ Cualitativa  ┆ Estado conyugal de mujer: casa… │
│ sufrio_violencia_pareja ┆ Cualitativa  ┆ Indica si sufrio violencia. 1 … │
│ factor_expansion        ┆ Cuantitativa ┆ Representa el número de mujere… │
└─────────────────────────┴──────────────┴────────────────────

In [29]:
# Crea DataFrame con dos columnas numéricas (edad_u_num e hijos_num)
df_union_valid = df.with_columns(
    # Convierte el texto de años e hijos a decimales  
    # strict=False: para convertir datos inválidos en null en lugar
    pl.col("edad_primer_union").cast(pl.Float64, strict=False).alias("edad_u_num"),
    pl.col("num_hijos").cast(pl.Float64, strict=False).alias("hijos_num")
).filter(
    # Descarta nulls y ponemos filtros .
    (pl.col("edad_u_num") >= 10) & (pl.col("edad_u_num") < 90) &
    (pl.col("hijos_num") >= 0) & (pl.col("hijos_num") < 20)
)

# Ejecuta consultas agregadas con Polars:
resumen = df_union_valid.select([
    # Calcula la media como "media_simple_edad"
    pl.col("edad_u_num").mean().alias("media_simple_edad"),
    # Calcula la mediana como "mediana_edad"
    pl.col("edad_u_num").median().alias("mediana_edad"),
    # Calcula cuartiles Q1 (percentil 25) y Q3 (percentil 75).
    pl.col("edad_u_num").quantile(0.25).alias("Q1_edad"),
    pl.col("edad_u_num").quantile(0.75).alias("Q3_edad"),
    # Calcula la media como "media_simple_hijos"
    pl.col("hijos_num").mean().alias("media_simple_hijos"),
    # Calcula la mediana como "mediana_hijos"
    pl.col("hijos_num").median().alias("mediana_hijos")
])

# Convierte los datos a arreglos unidimensionales de NumPy 
edades = df_union_valid["edad_u_num"].to_numpy()
hijos = df_union_valid["hijos_num"].to_numpy()
pesos = df_union_valid["factor_expansion"].to_numpy()

# Aplica la fórmula de la media ponderada (Paso 6), multiplica cada observación por su 
# factor de expansión y lo divide entre la suma de todos los factores de expansión. 
media_pond_edad = np.average(edades, weights=pesos)
media_pond_hijos = np.average(hijos, weights=pesos)

print(resumen)
# Imprime la edad promedio de la primera union (2 decimales)
print(f"\nMedia ponderada de edad a la primera unión: {media_pond_edad:.2f} años")
# IMprime el numero de hijos nacidos promedio (2 decimales)
print(f"Media ponderada de número de hijos: {media_pond_hijos:.2f}")

shape: (1, 6)
┌───────────────────┬──────────────┬─────────┬─────────┬────────────────────┬───────────────┐
│ media_simple_edad ┆ mediana_edad ┆ Q1_edad ┆ Q3_edad ┆ media_simple_hijos ┆ mediana_hijos │
│ ---               ┆ ---          ┆ ---     ┆ ---     ┆ ---                ┆ ---           │
│ f64               ┆ f64          ┆ f64     ┆ f64     ┆ f64                ┆ f64           │
╞═══════════════════╪══════════════╪═════════╪═════════╪════════════════════╪═══════════════╡
│ 20.581874         ┆ 18.0         ┆ 13.0    ┆ 25.0    ┆ 2.7227             ┆ 3.0           │
└───────────────────┴──────────────┴─────────┴─────────┴────────────────────┴───────────────┘

Media ponderada de edad a la primera unión: 20.34 años
Media ponderada de número de hijos: 2.67


In [33]:

# Función que recibe un subgrupo de datos y el nombre de la variable a analizar.
def calcular_variabilidad(df_grupo, col_name, nombre_grupo):
    # Extrae la columna elegida como un arreglo numérico de NumPy.
    datos = df_grupo[col_name].to_numpy()
    # Calcula el promedio del grupo
    mean_val = np.mean(datos)
    # Calcula la desviación estándar muestral (Paso 7)
    std = np.std(datos, ddof=1)
    # Obtiene cuartiles 1 y 3 con np.percentile()
    q1 = np.percentile(datos, 25)
    q3 = np.percentile(datos, 75)
    
    return {
        "Grupo": nombre_grupo,
        "Media": mean_val,
        "Desviación Estándar (s)": std,
        # Eleva al cuadrado la desviación estándar  para obtener varianza muestral 
        "Varianza (sxs)": std**2,
        # Obtenemos el Rango Intercuartílico con q3 - q1
        "IQR": q3 - q1,
        # Calcula Coeficiente de Variación dividiendo desviación estándar entre la media 
        # y multiplica por 100 para obtener el porcentaje de dispersión relativa
        "CV (%)": (std / mean_val) * 100
    }

# Separamos entre quien sufrio violencia y quien no (1 = SI, 0 = No)
grupo_con = df_union_valid.filter(pl.col("sufrio_violencia_pareja") == 1)
grupo_sin = df_union_valid.filter(pl.col("sufrio_violencia_pareja") == 0)

# Llamamos a la función para obtener media, desviación estándar, varianza, IQR y CV de la edad.
metrica_con = calcular_variabilidad(grupo_con, "edad_u_num", "Con Violencia")
metrica_sin = calcular_variabilidad(grupo_sin, "edad_u_num", "Sin Violencia")

# 3. Crear el DataFrame de Polars
df_comparativo = pl.DataFrame([metrica_con, metrica_sin])

print("--- COMPARACIÓN DE DISPERSIÓN DE EDAD A LA PRIMERA UNIÓN ---")
df_comparativo

--- COMPARACIÓN DE DISPERSIÓN DE EDAD A LA PRIMERA UNIÓN ---


Grupo,Media,Desviación Estándar (s),Varianza (sxs),IQR,CV (%)
str,f64,f64,f64,f64,f64
"""Con Violencia""",20.32715,9.346883,87.364223,12.0,45.982261
"""Sin Violencia""",20.725402,9.738521,94.838789,13.0,46.988332
